# Using Layout With CSS

Since panel is our main method for rendering plots and visuals with tsvi, it's worthwhile exploring what we can do with it especially in terms of webpage design. In web development, CSS is the language that governs how a page's content is rendered in a browser. We can use this to create some professional and clean graphics with our MT data.

Let's start with a few imports again. If you have been following along in the previous tutorials, the only package that may be missing here is pendulum, so we'll do a pip call to fix this first:

In [1]:
!pip install panel param pendulum

In [2]:
import panel as pn
import param
import pendulum

Next we'll make a multi-string literal and define some basic CSS attributes for our panel. In this case, we'll just start our header at the top of the page and make it so it is always visible no matter where we scroll:

In [3]:
CSS = '''
.header {
    position: sticky;
    top: 0;
}
'''

And then load them:

In [4]:
pn.extension(raw_css=[CSS])

Of course, having a layout for a webpage is not that impressive unless there are things to put in it. One way to keep all of our work tidy and readable is to use param.ClassSelector() to load a set of configuration variables, and then pass that to a small class containing all of our panel calls. 

To show what we're talking about, here's a quick GlobalConfig class containing some basic widgets, indicators, and a couple dictionaries for storing our data sources and the plots generated from them:

In [5]:
class GlobalConfig(param.Parameterized):
    title = param.String(default="Timeseries Viewer", doc="Title of the app.")
    
    timefilter = pn.widgets.DatetimeRangePicker(end=None, start=None, name='Time Filter')
    timezone = pn.widgets.FloatInput(value=0, name='Timezone', width=75, start=-12, end=14, step=1, mode='float', placeholder='Select a timezone.')
    
    loading_spinner =  pn.indicators.LoadingSpinner(width=40, height=40)
    
    sources = param.Dict(doc="Dict of datasources and it's configurations.")
    plots = param.Dict(doc="Dict of plots and it's configurations.")

Now we can create a Dashboard class containing all of the components of our window. All of this is really a glorified set of panel invocations. Then all that's left to do is to initialize a GlobalConfig object and pass it to the Dashboard constructor:

In [6]:
class Dashboard(param.Parameterized):
    
    global_config = param.ClassSelector(class_ = GlobalConfig)
    config_layout = pn.FlexBox(flex_direction='row')
    
    plot_layout = pn.FlexBox(flex_direction='column')
    
    @property
    def config_layout(self):
        return pn.FlexBox(self.global_config.timefilter,  self.global_config.timezone,  self.global_config.loading_spinner, flex_direction='row', css_classes=['.header'])
    
    @param.depends('global_config.timefilter.value', 'global_config.timezone.value', watch=True)
    def test(self):
        print(f'global config has changed! {self.global_config.timefilter}')
        
    def add_plot(self, source, variable):
        pass

In [13]:
gc = GlobalConfig()
layout = Dashboard(global_config = gc)

Because FlexBox objects are dynamic, we can change the options either before or after we render them to observe the results. For instance, we can change the width of the box, view it, then edit the content without needing to invoke ``layout.config_layout`` again.

In [14]:
layout.config_layout.width=1500

In [15]:
layout.config_layout

global config has changed! DatetimeRangePicker(as_numpy_datetime64=False, name='Time Filter') 
global config has changed! DatetimeRangePicker(as_numpy_datetime64=False, name='Time Filter')

FlexBox(css_classes=['.header'], objects=[DatetimeRangePicker(as_nu...], sizing_mode='stretch_width')

In [16]:
layout.global_config.loading_spinner.value = False

There are all sorts of widgets we can add via panel to layouts. Here's how you can create a dropdown menu containing a list of timezones:

In [23]:
pn.widgets.Select(options=list(pendulum.tz.timezones()))

<class 'functools._lru_cache_wrapper'>


Select(options=['Asia/Rangoon', ...], value='Asia/Rangoon')

The way our GlobalConfig class is set up, you can use ``param.update()`` to update whatever attributes you need to simultaneously:

In [30]:
gc.param.update(plots={'a': 2, 'b': 3})

In [32]:
gc.param.trigger('plots')

In [34]:
layout = pn.FlexBox(flex_direction='column')

Finally, by using the ``doc`` attribute in the configuration class, we can provide helpful tooltips for users that they can cursor over:

In [38]:
gc.title = 'Sample Title'
layout.append(gc.param.title)
layout